<a href="https://colab.research.google.com/github/SandeepKonduruFeb12/aiml/blob/master/gold/GoldAssignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


The QLoRA fine-tuning process, highlighting the key components.

## Setup Environment

Install all necessary Python libraries for QLoRA fine-tuning, including `transformers`, `peft`, `bitsandbytes`, `accelerate`, and `trl`.


In [1]:
pip install transformers peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 kB 40.6 MB/s eta 0:00:00


## Load Base Model and Tokenizer
Load a pre-trained open-source Mistral model (e.g., Mistral-7B) and its corresponding tokenizer from Hugging Face Hub, configuring it for 4-bit quantization using `BitsAndBytesConfig`.


**Reasoning**:
The first step in loading the model and tokenizer is to import the necessary libraries. This includes `AutoModelForCausalLM` and `AutoTokenizer` from `transformers`, `BitsAndBytesConfig` from `bitsandbytes`, and `torch` for specifying data types for quantization.



In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Necessary libraries imported.")

Necessary libraries imported.



Now that the necessary libraries are imported, Lets define the 4-bit quantization configuration, then load the pre-trained Mistral model with this configuration, and finally load its corresponding tokenizer.



In [3]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_id = "mistralai/Mistral-7B-v0.1"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Model and Tokenizer loaded with 4-bit quantization.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model and Tokenizer loaded with 4-bit quantization.


## Prepare Model for QLoRA Fine-tuning

Prepare the loaded 4-bit quantized model for QLoRA fine-tuning by configuring LoRA adapters and enabling gradient checkpointing.

#### Steps
1. Import necessary components: `LoraConfig` and `get_peft_model` from `peft`, and `prepare_model_for_kbit_training` from `peft.tuners.lora`.
2. Enable gradient checkpointing for the model to save memory during training.
3. Prepare the model for k-bit training using `prepare_model_for_kbit_training`.
4. Define the `LoraConfig` with appropriate parameters (e.g., `r`, `lora_alpha`, `target_modules`, `lora_dropout`, `bias`, `task_type`).
5. Apply the LoRA configuration to the model using `get_peft_model`.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 2. Enable gradient checkpointing
model.gradient_checkpointing_enable(use_reentrant=False)

# 3. Prepare the model for k-bit training
model = prepare_model_for_kbit_training(model)

# 4. Define the LoraConfig
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "lm_head"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 5. Apply the LoRA configuration to the model
model = get_peft_model(model, lora_config)

print("Model prepared for QLoRA fine-tuning with LoRA adapters and gradient checkpointing enabled.")
print(model.print_trainable_parameters())

In [5]:
from datasets import load_dataset

print("datasets library imported.")

datasets library imported.



Now that the `load_dataset` function is imported, Lets proceed to load the `yahma/alpaca-cleaned` dataset, which is a suitable instruction-following dataset from the Hugging Face Hub.



In [6]:
dataset = load_dataset("yahma/alpaca-cleaned")
print("Dataset loaded successfully.")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Dataset loaded successfully.
DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})


In [7]:
print("Dataset column names:", dataset['train'].column_names)
print("Dataset features:", dataset['train'].features)
print("\nFirst 3 examples of the training dataset:")
for i in range(3):
    print(f"--- Example {i+1} ---")
    print(f"Instruction: {dataset['train'][i]['instruction']}")
    print(f"Input: {dataset['train'][i]['input']}")
    print(f"Output: {dataset['train'][i]['output']}")

Dataset column names: ['output', 'input', 'instruction']
Dataset features: {'output': Value('string'), 'input': Value('string'), 'instruction': Value('string')}

First 3 examples of the training dataset:
--- Example 1 ---
Instruction: Give three tips for staying healthy.
Input: 
Output: 1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune functi

**Reasoning**:
Now that the dataset structure is understood, I will define a formatting function to convert each example into a single string prompt suitable for instruction tuning, concatenating the instruction, input, and output fields.



In [8]:
def format_instruction_tune_prompt(example):
    instruction = example['instruction']
    input_text = example['input']
    output_text = example['output']

    if input_text:
        prompt = f"### Instruction:\n{instruction}\n### Input:\n{input_text}\n### Output:\n{output_text}"
    else:
        prompt = f"### Instruction:\n{instruction}\n### Output:\n{output_text}"
    return {"text": prompt}

print("Formatting function 'format_instruction_tune_prompt' defined.")

Formatting function 'format_instruction_tune_prompt' defined.


**Reasoning**:
Now that the formatting function `format_instruction_tune_prompt` is defined, I will apply it to the dataset to create a new column named 'text' containing the formatted prompts for instruction tuning, as per the subtask instructions.



In [9]:
tokenized_dataset = dataset.map(format_instruction_tune_prompt, remove_columns=['instruction', 'input', 'output'])
print("Dataset formatted and tokenized successfully.")
print(tokenized_dataset['train'][0]['text'])

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

Dataset formatted and tokenized successfully.
### Instruction:
Give three tips for staying healthy.
### Output:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


**Reasoning**:
To complete the subtask, I will now split the dataset into training and evaluation sets for demonstration purposes. I will use a small subset of the training data as the training set and a small subset as the evaluation set.



In [10]:
train_dataset = tokenized_dataset['train'].select(range(1000)) # Use a small subset for training
eval_dataset = tokenized_dataset['train'].select(range(1000, 1100)) # Use a small subset for evaluation

print("Dataset split into training and evaluation sets successfully.")
print(f"Training set size: {len(train_dataset)}")
print(f"Evaluation set size: {len(eval_dataset)}")

Dataset split into training and evaluation sets successfully.
Training set size: 1000
Evaluation set size: 100


## Set Up SFTTrainer

### Subtask:
Initialize the SFTTrainer from the trl library, passing in the quantized model, tokenizer, prepared dataset, LoRA configuration, and TrainingArguments (e.g., batch size, learning rate, number of epochs).


In [12]:
import trl
from trl import SFTTrainer
from transformers import TrainingArguments

# 2. Define the TrainingArguments
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    bf16=True, # Changed from fp16=True to bf16=True
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    logging_steps=25,
    save_steps=25,
    eval_strategy="steps",
    eval_steps=25,
    report_to="none"
)

# 3. Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_arguments,
)

print("SFTTrainer initialized successfully.")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

SFTTrainer initialized successfully.


## Fine-tune the Model

### Subtask:
Execute the fine-tuning process using the `trainer.train()` method. This will apply QLoRA to the base model using the provided dataset.


**Reasoning**:
With the `SFTTrainer` correctly initialized after adjusting the `TrainingArguments` to use `bf16=True`, I will now execute the `trainer.train()` method to begin the QLoRA fine-tuning process.



In [14]:
trainer.train()
print("Fine-tuning process started.")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
25,0.818600,1.038234
50,1.038200,1.004639
75,0.908800,1.004431
100,0.958000,1.001288
125,0.955700,1.002879


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to

Fine-tuning process started.


## Demonstrate Before and After Fine-tuning

To show the effect of fine-tuning, we will generate text using both the original base model and the fine-tuned model with the same input prompt. This will highlight how the model's behavior and responses have changed.

In [15]:
test_prompt = "### Instruction:\nExplain the concept of quantum entanglement.\n### Output:"

print(f"Test Prompt:\n{test_prompt}\n")

Test Prompt:
### Instruction:
Explain the concept of quantum entanglement.
### Output:



### Response from Original Base Model

First, let's load the original `Mistral-7B-v0.1` model (without any fine-tuning or LoRA adapters) to see its response to the prompt.

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the original base model and tokenizer without quantization or PEFT
original_model_id = "mistralai/Mistral-7B-v0.1"
original_tokenizer = AutoTokenizer.from_pretrained(original_model_id)
original_model = AutoModelForCausalLM.from_pretrained(original_model_id, device_map="auto")

# Set pad_token_id to eos_token_id if it's not set
if original_tokenizer.pad_token_id is None:
    original_tokenizer.pad_token_id = original_tokenizer.eos_token_id

# Encode the prompt
input_ids = original_tokenizer(test_prompt, return_tensors="pt").input_ids.cuda()

# Generate text from the original model
original_outputs = original_model.generate(
    input_ids=input_ids,
    max_new_tokens=256,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7,
    eos_token_id=original_tokenizer.eos_token_id
)

original_response = original_tokenizer.decode(original_outputs[0], skip_special_tokens=True)
print("Original Model's Response:")
print(original_response)

# Clear memory
del original_model
import gc
gc.collect()
torch.cuda.empty_cache()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


KeyboardInterrupt: 

### Response from Fine-tuned Model

Now, let's use the fine-tuned model to generate a response to the same prompt.

In [ ]:
# Set pad_token_id for the fine-tuned tokenizer as well
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Encode the prompt for the fine-tuned model
input_ids_tuned = tokenizer(test_prompt, return_tensors="pt").input_ids.cuda()

# Generate text from the fine-tuned model
tuned_outputs = model.generate(
    input_ids=input_ids_tuned,
    max_new_tokens=256,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7,
    eos_token_id=tokenizer.eos_token_id
)

tuned_response = tokenizer.decode(tuned_outputs[0], skip_special_tokens=True)
print("Fine-tuned Model's Response:")
print(tuned_response)
